In [ ]:
import method_utils as mu




def get_dsaids_w_umls()-> set:
    """
    Get a set of DO IDs that have UMLS codes associated with them.
    """
    dsaids = set()

    return dsaids


do_g = mu.get_do_graph()

umls_2_doid = mu.get_umls_2_doid_mapping(do_g)


# load DSAIDs
dsaids = mu.get_dsaids()

In [ ]:
import torch
import numpy as np
CLS = False  # Single-label classification
CLS_MULTILABEL = True  # Multilabel classification

output_values = np.array([[0,0,1],[1,0,0]])  # Placeholder for model output
celltype_labels = np.array([2,0])  # Placeholder for true labels


output_values = torch.tensor([[-10, -10, 20], [20, -10, -10]])  # Placeholder for model output
disease_multilabel = torch.tensor([[1, 0, 1], [0, 1, 1]])  # Placeholder for multilabel true labels
input_gene_ids = torch.tensor([[1, 0, 1], [0, 1, 1]])
total_error = 0
total_num = 0
predictions = []

if CLS:
    # Standard single-label accuracy
    accuracy = (output_values.argmax(1) == celltype_labels).sum().item()
    total_error += (1 - accuracy / len(input_gene_ids)) * len(input_gene_ids)
elif CLS_MULTILABEL:
    # Multilabel: compute sigmoid + threshold
    preds = torch.sigmoid(output_values).numpy()
    preds_bin = (preds > 0.5).astype(int)
    true = disease_multilabel.numpy()

    # Sample-wise accuracy: proportion of labels correctly predicted
    samplewise_acc = (preds_bin == true).mean(axis=1).mean()
    total_error += (1 - samplewise_acc) * len(input_gene_ids)
    
total_num += len(input_gene_ids)


print(total_error / total_num)

TypeError: sigmoid(): argument 'input' (position 1) must be Tensor, not numpy.ndarray

In [26]:
output_values.argmax(1)

array([2, 0])

In [23]:
output_values.argmax(1)

array([2, 0])

In [7]:
import numpy as np
a = np.array([[1, 0, 1], [0, 1, 0], [1, 1, 0]])
b = np.array([[1, 0, 1], [0, 1, 0], [1, 0, 0]])

samplewise_acc = (a == b).mean(axis=1).mean()
(1 - samplewise_acc) * len(a)

0.3333333333333335

In [13]:
a = np.array([[1, 0, 1], [0, 1, 0], [1, 1, 0]])
b = np.array([[1, 0, 0], [0, 1, 0], [1, 0, 0]])

accuracy = (a.argmax(1) == b).sum().item()
total_error = (1 - accuracy / len(a)) * len(a)

print(total_error)

-2.0


In [17]:
(a.argmax(1) == b).sum().item()/len(a)

1.6666666666666667

In [8]:
samplewise_acc

0.8888888888888888

In [ ]:
# isolating function

# Load data

processed_ids = get_processed_ids()
logging.info(f"Nº of processed ids: {len(processed_ids)}")

df_data_info = pd.read_csv(data_info_path)

df_data_info_processed = df_data_info.copy()

df_data_info_processed = df_data_info_processed[
    df_data_info_processed["dsaid"].isin(processed_ids)
]

logging.info(f"Nº of processed ids in df_data_info: {len(df_data_info_processed)}")

df_data_info_processed_filtered = df_data_info_processed[
    (df_data_info_processed["organism"] == "Homo sapiens")
    & (
        (df_data_info_processed["library_strategy"] == "Microarray")
        | (df_data_info_processed["library_strategy"] == "RNA-Seq")
    )
]
logging.info(
    f"Nº of Filtered by library (filter out single cell): {df_data_info_processed_filtered.shape}"
)

# get all entrez protein-coding human ids
global human_entrez_protein_coding_ids

# Filter 1: Human Genes
human_dsaids = df_data_info_processed_filtered["dsaid"].to_list()

pct_thr = 0.5
# even though we have filtered by human, we now look at the actual genes! Some may not be human !
pct_human = list()
dsaids_f1 = list()
for i in range(len(signatures)):
    if signatures[i][0] in human_dsaids:
        _pct = len(
            set(signatures[i][2]).intersection(human_entrez_protein_coding_ids)
        ) / len(set(signatures[i][2]))
        pct_human.append(_pct)
        if _pct > pct_thr:
            dsaids_f1.append(signatures[i][0])
pct_human = np.array(pct_human)

df_data_info_processed_filtered = df_data_info_processed_filtered.copy()
df_data_info_processed_filtered["n_samples"] = [
    int(x.split("|")[1])
    for x in df_data_info_processed_filtered["control_case_sample_count"]
]

df_query = df_data_info_processed_filtered.query("dsaid in @dsaids_f1")

print(f"Nº of signatures: {len(df_query)}")
print(f"Nº of datasets : {len(df_query['accession'].unique())}")
print(f"Nº of diseases : {len(df_query['disease'].unique())}")
print(f"Nº of samples : {df_query['n_samples'].sum()}")

# Filter 2: Type of Sequencing
df_data_info_processed_filtered["library_strategy"].value_counts()
dsaids_f2 = dsaids_f1

# Filter 3: Presence of Disease Ontology IDs
# Leaf nodes Diseases
leaf_nodes = [
    node
    for node in do_G.nodes()
    if do_G.in_degree(node) == 0 and do_G.out_degree(node) > 0
]

dsaids_w_doids = [d for d in dsaids_f2 if d in dsaids_2_doids.keys()]
dsaids_w_diseases = [
    d for d in dsaids_w_doids if len(set(dsaids_2_doids[d]) & set(leaf_nodes)) > 0
]

# for now only take DSAIDS with 1 LEAF DISEASE!
dsaids_f3 = [
    d
    for d in dsaids_w_diseases
    if len(set(dsaids_2_doids[d]) & set(leaf_nodes)) == 1
]
print(f"Nº of DSAIDS with 1 leaf disease {len(dsaids_f3)}")

# Filter 3.1: Presence of Disease Ontology IDs
flatten = lambda l: [item for sublist in l for item in sublist]
df_query = df_data_info_processed_filtered.query("dsaid in @dsaids_w_doids")
print(f"Filtering 3.1 - has doid")
print(f"Nº of signatures: {len(df_query)}")
print(f"Nº of datasets : {len(df_query['accession'].unique())}")
print(f"Nº of diseases : {len(df_query['disease'].unique())}")
print(
    f"Nº of disease ontology ids: {len(set(flatten([dsaids_2_doids[x] for x in dsaids_w_doids])))}"
)
print(
    f"Nº of disease ontology diseases: {len(set(flatten([list(set(dsaids_2_doids[x])&set(leaf_nodes)) for x in dsaids_w_doids])))}"
)
print(f"Nº of samples : {df_query['n_samples'].sum()}")

# Filter 3.2: Presence of Leaf Disease Ontology IDs
df_query = df_data_info_processed_filtered.query("dsaid in @dsaids_w_diseases")
print(f"Filtering 3.2 - has doid leaf")
print(f"Nº of signatures: {len(df_query)}")
print(f"Nº of datasets : {len(df_query['accession'].unique())}")
print(f"Nº of diseases : {len(df_query['disease'].unique())}")
print(
    f"Nº of disease ontology ids: {len(set(flatten([dsaids_2_doids[x] for x in dsaids_w_diseases])))}"
)
print(
    f"Nº of disease ontology diseases: {len(set(flatten([list(set(dsaids_2_doids[x])&set(leaf_nodes)) for x in dsaids_w_diseases])))}"
)
print(f"Nº of samples : {df_query['n_samples'].sum()}")

# Filter 3.3: Presence of 1 Leaf Disease Ontology IDs
df_query = df_data_info_processed_filtered.query("dsaid in @dsaids_f3")
print(f"Filtering 3.3 - has 1 doid leaf")
print(f"Nº of signatures: {len(df_query)}")
print(f"Nº of datasets : {len(df_query['accession'].unique())}")
print(f"Nº of diseases : {len(df_query['disease'].unique())}")
print(
    f"Nº of disease ontology ids: {len(set(flatten([dsaids_2_doids[x] for x in dsaids_f3])))}"
)
print(
    f"Nº of disease ontology diseases: {len(set(flatten([list(set(dsaids_2_doids[x])&set(leaf_nodes)) for x in dsaids_f3])))}"
)
print(f"Nº of samples : {df_query['n_samples'].sum()}")

# Filter 4: Presence of DE Genes
f_signatures = [s for s in signatures if s[0] in dsaids_f3]
de_genes = process_map(get_de_genes, f_signatures, max_workers=8, chunksize=10)

# Filter 4.1: Presence of DE Genes
# Less than 50% of genes DE
# at least 50 DE genes
mask_de_genes = np.array(
    [(True if 0 <= len(d[0]) + len(d[1]) <= 1 * d[2] else False) for d in de_genes]
)

# get dsaids which pass filter
dsaids_f4 = np.array(dsaids_f3)[mask_de_genes]
print(f"Filtered by nº of DE genes: {len(dsaids_f4)}")

df_query = df_data_info_processed_filtered.query("dsaid in @dsaids_f4")
print(f"Filtering 4 - has 1 doid leaf")
print(f"Nº of signatures: {len(df_query)}")
print(f"Nº of datasets : {len(df_query['accession'].unique())}")
print(f"Nº of diseases : {len(df_query['disease'].unique())}")
print(
    f"Nº of disease ontology ids: {len(set(flatten([dsaids_2_doids[x] for x in dsaids_f4])))}"
)
print(
    f"Nº of disease ontology diseases: {len(set(flatten([list(set(dsaids_2_doids[x])&set(leaf_nodes)) for x in dsaids_f4])))}"
)
print(f"Nº of samples : {df_query['n_samples'].sum()}")